# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\nDescription: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
record_sets_metadata = dataset.record_sets

print('Available record sets:')
for rs in record_sets_metadata:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, print available fields and their @id
for rs in record_sets_metadata:
    print(f"\nRecord set: {rs.get('name', 'N/A')} (@id: {rs['@id']})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        print(f"  - Field @id: {f['@id']} | name: {f.get('name', 'N/A')} | dataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Select record sets by their @id
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set @id: {record_set_id}")

# Example: show the columns and head for the first record set
example_record_set_id = record_sets_ids[0]
print(f"\nColumns of record set @id {example_record_set_id}:")
print(dataframes[example_record_set_id].columns.tolist())
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prep for further analysis.

In [ ]:
# Choose a record set and fields for EDA
# Example: Assume the main survey records set is the first one
survey_record_set_id = record_sets_ids[0]
df = dataframes[survey_record_set_id]

# List numeric fields available
rs_def = next(rs for rs in dataset.record_sets if rs['@id'] == survey_record_set_id)
numeric_fields = [f for f in rs_def.get('field', []) if f.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]['@id']
    numeric_field_name = numeric_fields[0]['name'] if 'name' in numeric_fields[0] else numeric_field_id
    print(f"Numeric field chosen: {numeric_field_name} (@id: {numeric_field_id})")

    threshold = df[numeric_field_name].mean()
    filtered_df = df[df[numeric_field_name] > threshold]
    print(f"\nFiltered records with {numeric_field_name} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_name}_normalized"] = (
        filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()
    ) / filtered_df[numeric_field_name].std()
    print(f"\nNormalized {numeric_field_name} for filtered records:")
    print(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

    # Group by a categorical field
    group_fields = [f for f in rs_def.get('field', []) if f.get('dataType')=='schema:Text']
    if group_fields:
        group_field_id = group_fields[0]['@id']
        group_field_name = group_fields[0]['name'] if 'name' in group_fields[0] else group_field_id
        if group_field_name in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_name)[numeric_field_name].mean().reset_index()
            print(f"\nGrouped data by {group_field_name} (@id: {group_field_id}):")
            print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_name].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_name}')
    plt.xlabel(numeric_field_name)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot by group field
if numeric_fields and group_fields:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_name, y=numeric_field_name, data=df)
    plt.title(f'{numeric_field_name} by {group_field_name}')
    plt.xlabel(group_field_name)
    plt.ylabel(numeric_field_name)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from dataset exploration.

* Using the `mlcroissant` library, we loaded and explored the FAIR² Mental Health Survey dataset from Kilifi County, Kenya.
* Reviewed metadata, available record sets and fields using their `@id`s for precise referencing.
* Extracted survey data, filtered and normalized numeric fields, and grouped by demographic attributes.
* Visualized numeric field distributions and their relationships to demographic categories.
* This workflow demonstrates FAIR data principles and AI-readiness using Croissant schemas.